# 🎙️ Linear Predictive Coding (LPC) Speech Analysis
## Acoustic Phonetics Project: Minimal Pair Analysis ('कहा' vs 'कहाँ')
### Member 2: Linear Predictive Coding (LPC) & Inverse Filtering

---

### 📌 Project Context & Member 2 Role:
- **Member 1 (Audio Acquisition)**: Records and prepares speech `.wav` files.
- **Member 2 (YOU - LPC Analysis)**: Estimates the vocal-tract all-pole filter model, computes LPC coefficients, and performs inverse filtering to extract the **LPC residual signal**.
- **Member 3 (Phase & Timing)**: Analyzes phase coherence, temporal variations, and glottal pulse timing on the residual.
- **Member 4 (Classification & Comparison)**: Classifies and differentiates speech sounds using the extracted features.

### 🎯 Deliverables Produced by This Notebook:
1. `lpc_coefficients.npy` — Estimated vocal tract filter weights for each frame
2. `lpc_residual.npy` — Numerical prediction error / excitation signal
3. `lpc_residual.wav` — Auditory residual file for playback and inspection

In [ ]:
# Import required libraries
import os
import numpy as np
import matplotlib.pyplot as plt
import scipy.io.wavfile as wavfile
from IPython.display import Audio, display

from lpc_core.preprocessor import load_and_preprocess_audio, save_wav_file
from lpc_core.lpc_analyzer import extract_lpc_coefficients, levinson_durbin
from lpc_core.inverse_filter import compute_lpc_residual
from lpc_core.formant_analyzer import extract_formants_from_lpc, compute_lpc_spectrum
from lpc_core.features import extract_lpc_features
from lpc_core.visualizer import plot_waveform_and_residual, plot_spectrogram_comparison, plot_lpc_spectral_envelope

print("✅ LPC Analysis Environment Initialized Successfully!")

## 1. Load and Condition Speech Audio
- Ensures 16,000 Hz sampling rate
- Converts stereo to mono
- Removes DC offset bias
- Normalizes amplitude
- Applies high-pass pre-emphasis filter ($y[n] = x[n] - 0.97 x[n-1]$) to boost high-frequency formants

In [ ]:
audio_file = "data/kaha.wav"
proc_signal, raw_signal, fs = load_and_preprocess_audio(audio_file, target_fs=16000, apply_preemph=True)

print(f"Audio file: {audio_file}")
print(f"Sampling Rate: {fs} Hz")
print(f"Total Samples: {len(raw_signal):,}")
print(f"Duration: {len(raw_signal) / fs:.3f} seconds")

# Play Original Audio
print("Listen to Original Audio:")
display(Audio(raw_signal, rate=fs))

## 2. Frame-by-Frame LPC Estimation (Levinson-Durbin)
Because vocal tract geometry changes dynamically as the articulators move, speech is quasi-stationary only over short intervals.
- **Frame length**: 25 ms (400 samples at 16 kHz)
- **Hop length**: 10 ms (160 samples at 16 kHz)
- **LPC Order ($p$)**: 16 (captures vocal tract resonant poles up to 8 kHz)
- **Window**: Hamming window to reduce spectral leakage

In [ ]:
lpc_order = 16
lpc_res = extract_lpc_coefficients(
    proc_signal,
    fs=fs,
    order=lpc_order,
    frame_ms=25.0,
    hop_ms=10.0,
    window_type="hamming"
)

lpc_coeffs = lpc_res['lpc_coeffs']
print(f"Number of frames: {lpc_coeffs.shape[0]}")
print(f"Coefficients per frame: {lpc_coeffs.shape[1]} (including leading 1.0 for A(z))")
print(f"First frame coefficients sample: {np.round(lpc_coeffs[10][:6], 4)}")

## 3. Inverse Filtering & LPC Residual Extraction
The LPC residual $e[n]$ is the prediction error:
$$e[n] = x[n] - \hat{x}[n] = x[n] + \sum_{k=1}^p a_k x[n-k]$$
Inverse filtering applies the FIR filter $A(z) = 1 + \sum_{k=1}^p a_k z^{-k}$ to undo the vocal tract resonances, leaving predominantly the excitation signal.

In [ ]:
# Calculate full residual signal
residual = compute_lpc_residual(proc_signal, lpc_res, method="sample_varying")

print(f"Residual signal computed! Length: {len(residual):,} samples")

# Plot Waveform vs Residual
fig1 = plot_waveform_and_residual(raw_signal, residual, fs, title="'kaha' (कहा)")
plt.show()

# Listen to Residual Audio
norm_res = residual / (np.max(np.abs(residual)) + 1e-9) * 0.9
print("Listen to LPC Residual Audio (Vocal-tract resonances stripped away):")
display(Audio(norm_res, rate=fs))

## 4. Spectrogram & Formant Envelope Analysis
Observe how the dense formant bands in the speech spectrogram are flattened in the LPC residual spectrogram, verifying successful inverse filtering.

In [ ]:
# Spectrogram Comparison
fig2 = plot_spectrogram_comparison(raw_signal, residual, fs, title="Spectrogram Analysis")
plt.show()

# Spectral Envelope vs FFT Harmonics in a voiced vowel frame
mid_idx = len(lpc_coeffs) // 2
frame_start = lpc_res['indices'][mid_idx]
mid_frame = proc_signal[frame_start : frame_start + lpc_res['frame_len']]

fig3 = plot_lpc_spectral_envelope(mid_frame, lpc_coeffs[mid_idx], fs, frame_time=lpc_res['frame_times'][mid_idx])
plt.show()

formants, _ = extract_formants_from_lpc(lpc_coeffs[mid_idx], fs=fs)
print("Extracted Vocal Tract Formants (Resonances):")
for i, f in enumerate(formants[:4]):
    print(f"  -> Formant F{i+1}: {int(f)} Hz")

## 5. Comparative Study: Oral 'kaha' vs Nasalized 'kahaan'
Demonstrates how the LPC model and residual distinguish the Hindi acoustic minimal pair.

In [ ]:
from compare_words import compare_pair

report = compare_pair("data/kaha.wav", "data/kahaan.wav", name1="kaha", name2="kahaan")
print("\nSummary for Member 4:")
print(f"Mean LPC Cosine Distance: {report['mean_lpc_cosine_distance']}")
print(f"Residual Energy Ratio (kahaan / kaha): {report['residual_rms_ratio']}")

## 6. Save Artifacts for Member 3 and Member 4
Exporting the standard files required by the team workflow.

In [ ]:
os.makedirs("outputs/kaha", exist_ok=True)

np.save("outputs/kaha/lpc_coefficients.npy", lpc_coeffs)
np.save("outputs/kaha/lpc_residual.npy", residual)
save_wav_file("outputs/kaha/lpc_residual.wav", norm_res, fs=fs)

print("📁 Files Saved Successfully:")
print("  1. outputs/kaha/lpc_coefficients.npy", os.path.exists("outputs/kaha/lpc_coefficients.npy"))
print("  2. outputs/kaha/lpc_residual.npy", os.path.exists("outputs/kaha/lpc_residual.npy"))
print("  3. outputs/kaha/lpc_residual.wav", os.path.exists("outputs/kaha/lpc_residual.wav"))